# FloodShift 05 — Robustness, Uncertainty Calibration & Abstention

This notebook begins **after segmentation development has closed**.

The frozen primary segmenter is:

\[
\boxed{\text{04H epoch-4 weights} + \text{fixed water-type bias } b=-0.15}
\]

with selected validation performance:

- mIoU: **0.796029**
- non-flood IoU: **0.976510**
- permanent-water IoU: **0.825058**
- temporary-flood IoU: **0.586519**

## Purpose

Notebook 05 does **not** train or alter the segmentation network.

It does four things:

1. **Integrity reproduction**  
   Reproduce the frozen 04H-CAL validation result exactly enough to verify the correct checkpoint/config.

2. **Confidence calibration**  
   Fit one scalar temperature \(T\) on a deterministic calibration half of the existing validation scenes. Temperature scaling cannot change the segmentation argmax.

3. **Abstention policy**  
   Freeze entropy thresholds corresponding to approximately **10%** and **20%** review rates on clean calibration data.

4. **Robustness audit**  
   Evaluate the frozen model on the other deterministic half of validation under a predeclared set of sensor/reliability stress conditions.

## Important scientific boundary

The existing 68-scene validation split has already been used during model development, so Notebook 05 does **not** claim that its internal 34/34 partition is a new independent benchmark. It is used only to prevent the temperature/abstention parameters from being fitted and evaluated on exactly the same pixels.

The **58-scene Pakistan + Spain geographic test remains unopened**.

## Required Kaggle inputs

Attach:

1. The same permanently frozen preprocessing output used by 04H / 04H-CAL.
2. `FloodShift_04H_best_mIoU.pt`
   - expected SHA256:
     `ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53`
3. `FloodShift_04H_CAL_config.pt`
   - must reference the same 04H SHA
   - must contain `type_bias = -0.15`
   - must contain decision `04H_CAL_SINGLE_MODEL_PREFERRED`

No 04E, 04F, or 04G checkpoint is required.

---

## Predeclared robustness protocol

All corruptions are applied to **normalized model inputs** before TTA.

| Scenario | Definition |
|---|---|
| clean | unchanged input |
| optical_haze_proxy | 7×7 low-pass optical blur, 0.60 contrast-to-mean attenuation, reliability ×0.25 |
| optical_missing | optical tensor = 0, reliability = 0 |
| sar_noise_0p35 | additive Gaussian stress of 0.35 normalized SAR standard deviations on valid SAR pixels |
| sar_missing | SAR tensor = 0 |
| combined_mild | optical haze proxy + SAR noise of 0.25 normalized std |
| reliability_neutral | reliability prior forced to 0.5 |
| reliability_inverted | reliability prior replaced by \(1-P(\mathrm{clear})\) |

The Gaussian conditions are **standardized-domain stress tests**, not a claim that Gaussian noise is a physical SAR speckle model.

All scenarios use the same fixed 4-way TTA as the selected segmenter.


In [ ]:
# ============================================================
# CELL: P12-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P10-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E/04F-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P12-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P10-C01.")

In [1]:
# ============================================================
# CELL: P12-C01 — ENVIRONMENT + REPRODUCIBILITY
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, time, math, random, hashlib, zipfile
from pathlib import Path

import numpy as np
import scipy
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

from sklearn.metrics import roc_auc_score, average_precision_score

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_05_Robustness_Uncertainty")
BOOTSTRAP_ROOT = OUTPUT_ROOT / "bootstrap"
METRIC_ROOT = OUTPUT_ROOT / "metrics"
FIGURE_ROOT = OUTPUT_ROOT / "figures"

for p in [OUTPUT_ROOT, BOOTSTRAP_ROOT, METRIC_ROOT, FIGURE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P12-C01 — NOTEBOOK 05 ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("TerraTorch  :", getattr(terratorch, "__version__", "unknown"))
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


P12-C01 — NOTEBOOK 05 ENVIRONMENT
Python      : 3.12.13
NumPy       : 1.26.4
Pandas      : 2.3.3
SciPy       : 1.16.3
PyTorch     : 2.10.0+cu128
TerraTorch  : unknown
Device      : cuda:0
Visible GPUs: 1
GPU         : Tesla T4
AMP         : True
Output root : /kaggle/working/FloodShift_05_Robustness_Uncertainty


In [2]:
# ============================================================
# CELL: P12-C02 — LOCATE FROZEN INPUTS + VERIFY LINEAGE
# ============================================================
_file_cache = {}

def locate_by_saved_path(saved_path_or_name, required=True):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    key = str(saved_path_or_name)

    if key in _file_cache:
        return _file_cache[key]

    matches = []
    for root in [INPUT_ROOT, BOOTSTRAP_ROOT]:
        if root.exists():
            matches.extend(root.rglob(filename))

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not locate '{filename}' under /kaggle/input."
            )
        return None

    selected = sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]
    _file_cache[key] = selected
    return selected

FINAL_MANIFEST_PATH = locate_by_saved_path("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate_by_saved_path("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate_by_saved_path("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate_by_saved_path("train_raw_sar_scene_statistics.csv")

CHECKPOINT_04H = locate_by_saved_path("FloodShift_04H_best_mIoU.pt")
CAL_CONFIG_PATH = locate_by_saved_path("FloodShift_04H_CAL_config.pt")

EXPECTED_04H_SHA256 = "ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53"
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"
EXPECTED_TYPE_BIAS = -0.15
EXPECTED_CAL_DECISION = "04H_CAL_SINGLE_MODEL_PREFERRED"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

SHA_04H = sha256_file(CHECKPOINT_04H)
assert SHA_04H == EXPECTED_04H_SHA256, (
    "Wrong 04H checkpoint. "
    f"Expected {EXPECTED_04H_SHA256}, got {SHA_04H}"
)

cal_cfg = torch.load(CAL_CONFIG_PATH, map_location="cpu", weights_only=False)

assert cal_cfg["source_04h_checkpoint_sha256"] == EXPECTED_04H_SHA256
assert cal_cfg["source_04e_sha256"] == EXPECTED_04E_SHA256
assert abs(float(cal_cfg["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12
assert cal_cfg["decision"] == EXPECTED_CAL_DECISION

TYPE_BIAS = float(cal_cfg["type_bias"])

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 112)
print("P12-C02 — FROZEN INPUTS VERIFIED")
print("=" * 112)
print("Manifest       :", FINAL_MANIFEST_PATH)
print("Global lock    :", GLOBAL_LOCK_PATH)
print("04H checkpoint :", CHECKPOINT_04H)
print("04H SHA256     :", SHA_04H)
print("CAL config     :", CAL_CONFIG_PATH)
print("Type bias      :", TYPE_BIAS)
print("CAL decision   :", cal_cfg["decision"])
print("Preprocess lock:", preprocessing_lock["status"])
print("P12-C02 COMPLETE ✅")


P12-C02 — FROZEN INPUTS VERIFIED
Manifest       : /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_manifest_v1.csv
Global lock    : /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_GLOBAL_LOCK_v1.json
04H checkpoint : /kaggle/input/datasets/himavarshini2108/4hptfilefornow/FloodShift_04H_best_mIoU.pt
04H SHA256     : ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53
CAL config     : /kaggle/input/datasets/himavarshini2108/4hcalptfile/FloodShift_04H_CAL_config.pt
Type bias      : -0.15
CAL decision   : 04H_CAL_SINGLE_MODEL_PREFERRED
Preprocess lock: PERMANENTLY_FROZEN
P12-C02 COMPLETE ✅


In [3]:
# ============================================================
# CELL: P12-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P12-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

P12-C03 — DATA CONFIG
Train / Val / Test: 314 68 58
Optical bands      : ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
SAR mean [VV,VH]   : [-10.358468 -17.10712 ]
SAR std  [VV,VH]   : [4.161632  4.8134336]
Reliability        : frozen continuous P(clear)
Primary SAR input  : raw VV/VH


In [4]:
# ============================================================
# CELL: P12-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [5]:
# ============================================================
# CELL: P12-C05 — DETERMINISTIC INTERNAL VAL PARTITION + LOADERS
#
# Full validation remains available only for integrity reproduction.
# 34 scenes -> confidence calibration
# 34 scenes -> robustness/uncertainty evaluation
# Test split is NEVER wrapped in a Dataset/DataLoader.
# ============================================================
VAL_BATCH_SIZE = 2
NUM_WORKERS = 2

def stable_scene_hash(scene_id):
    token = ("FloodShift05|" + str(scene_id)).encode("utf-8")
    return hashlib.sha256(token).hexdigest()

val_partition = val_df[["scene_id"]].copy()
val_partition["partition_hash"] = val_partition["scene_id"].map(stable_scene_hash)
val_partition = val_partition.sort_values("partition_hash").reset_index(drop=True)

calibration_ids = set(val_partition.iloc[:34]["scene_id"].astype(str))
robustness_ids = set(val_partition.iloc[34:]["scene_id"].astype(str))

assert len(calibration_ids) == 34
assert len(robustness_ids) == 34
assert calibration_ids.isdisjoint(robustness_ids)

val_cal_df = val_df[val_df["scene_id"].astype(str).isin(calibration_ids)].copy().reset_index(drop=True)
val_robust_df = val_df[val_df["scene_id"].astype(str).isin(robustness_ids)].copy().reset_index(drop=True)

assert len(val_cal_df) == 34
assert len(val_robust_df) == 34

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

common_loader = dict(
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    persistent_workers=(NUM_WORKERS > 0),
)

val_full_loader = DataLoader(FixedEvalDataset(val_df, train_mode=False), **common_loader)
val_cal_loader = DataLoader(FixedEvalDataset(val_cal_df, train_mode=False), **common_loader)
val_robust_loader = DataLoader(FixedEvalDataset(val_robust_df, train_mode=False), **common_loader)

print("=" * 112)
print("P12-C05 — INTERNAL VALIDATION PARTITION")
print("=" * 112)
print("Full validation scenes :", len(val_df))
print("Calibration scenes     :", len(val_cal_df))
print("Robustness scenes      :", len(val_robust_df))
print("Frozen test scenes     :", len(test_df), "(LOCKED; no Dataset/DataLoader constructed)")
print("Full val batches       :", len(val_full_loader))
print("Calibration batches    :", len(val_cal_loader))
print("Robustness batches     :", len(val_robust_loader))
print("P12-C05 COMPLETE ✅")


P12-C05 — INTERNAL VALIDATION PARTITION
Full validation scenes : 68
Calibration scenes     : 34
Robustness scenes      : 34
Frozen test scenes     : 58 (LOCKED; no Dataset/DataLoader constructed)
Full val batches       : 136
Calibration batches    : 68
Robustness batches     : 68
P12-C05 COMPLETE ✅


In [6]:
# ============================================================
# CELL: P12-C06 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits

In [7]:
# ============================================================
# CELL: P12-C07 — REBUILD EXACT 04H + LOAD EPOCH-4 CHECKPOINT
# ============================================================
class HierarchicalResidualHead(nn.Module):
    def __init__(self, in_channels=16, hidden=48, max_delta=1.50):
        super().__init__()
        self.max_delta = float(max_delta)
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, base_prob, sar, optical, reliability):
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        raw = self.net(x).float()
        return self.max_delta * torch.tanh(raw / self.max_delta)

class FloodShift04H(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = HierarchicalResidualHead()
        for p in self.base.parameters():
            p.requires_grad = False
        self.base.eval()

    @property
    def backbone(self):
        return self.base.backbone

    def train(self, mode=True):
        super().train(mode)
        self.base.eval()
        self.head.train(mode)
        return self

    def forward(self, sar, optical, reliability, return_aux=False):
        with torch.no_grad():
            base_out = self.base(sar, optical, reliability, return_aux=True)
            base_logits = base_out["logits"].float()
            base_prob = torch.softmax(base_logits, dim=1)

        correction = self.head(
            base_prob.to(sar.dtype),
            sar,
            optical,
            reliability,
        )

        delta_water = correction[:, 0]
        delta_type = correction[:, 1]

        base_water_logit = torch.logsumexp(base_logits[:, 1:3], dim=1) - base_logits[:, 0]
        base_type_logit = base_logits[:, 2] - base_logits[:, 1]

        water_logit = base_water_logit + delta_water
        type_logit = base_type_logit + delta_type

        p_water = torch.sigmoid(water_logit)
        p_temp_given_water = torch.sigmoid(type_logit)

        p0 = 1.0 - p_water
        p1 = p_water * (1.0 - p_temp_given_water)
        p2 = p_water * p_temp_given_water

        probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
        log_probs = probs.log()

        if return_aux:
            return {
                "logits": log_probs,
                "probs": probs,
                "base_logits": base_logits,
                "base_prob": base_prob,
                "base_water_logit": base_water_logit,
                "base_type_logit": base_type_logit,
                "water_logit": water_logit,
                "type_logit": type_logit,
                "delta_water": delta_water,
                "delta_type": delta_type,
            }

        return log_probs

payload = torch.load(CHECKPOINT_04H, map_location="cpu", weights_only=False)

assert isinstance(payload, dict)
assert payload.get("base_sha256") == EXPECTED_04E_SHA256
assert int(payload.get("epoch", -1)) == 4
assert payload.get("tag") == "best_mIoU"

base = FloodShiftTerraMindRIFT(num_classes=NUM_CLASSES, pretrained_backbone=False)
model = FloodShift04H(base)

missing, unexpected = model.load_state_dict(payload["state_dict"], strict=False)
assert not missing, f"Missing keys: {missing[:10]}"
assert not unexpected, f"Unexpected keys: {unexpected[:10]}"

bad = [
    n for n, t in model.state_dict().items()
    if torch.is_floating_point(t) and not torch.isfinite(t).all()
]
assert not bad, f"Non-finite tensors: {bad[:10]}"

model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False

print("=" * 112)
print("P12-C07 — FROZEN 04H-CAL MODEL READY")
print("=" * 112)
print("Epoch          :", payload["epoch"])
print("Tag            :", payload["tag"])
print("Type bias      :", TYPE_BIAS)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("P12-C07 COMPLETE ✅")


INFO:root:Loaded weights for BLUE in position 0 of patch embed
INFO:root:Loaded weights for GREEN in position 1 of patch embed
INFO:root:Loaded weights for RED in position 2 of patch embed
INFO:root:Loaded weights for RED_EDGE_1 in position 3 of patch embed
INFO:root:Loaded weights for RED_EDGE_2 in position 4 of patch embed
INFO:root:Loaded weights for RED_EDGE_3 in position 5 of patch embed
INFO:root:Loaded weights for NIR_BROAD in position 6 of patch embed
INFO:root:Loaded weights for NIR_NARROW in position 7 of patch embed
INFO:root:Loaded weights for SWIR_1 in position 8 of patch embed
INFO:root:Loaded weights for SWIR_2 in position 9 of patch embed


P12-C07 — FROZEN 04H-CAL MODEL READY
Epoch          : 4
Tag            : best_mIoU
Type bias      : -0.15
Trainable params: 0
P12-C07 COMPLETE ✅


In [8]:
# ============================================================
# CELL: P12-C08 — COMMON METRICS + FROZEN 4-WAY TTA
# ============================================================
TTA_MODES = ["identity", "hflip", "vflip", "rot180"]

def confusion_from_predictions(pred, target):
    valid = target != IGNORE_INDEX
    target_v = target[valid].long()
    pred_v = pred[valid].long()

    indices = NUM_CLASSES * target_v + pred_v
    return torch.bincount(
        indices,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics_from_confusion(cm):
    if torch.is_tensor(cm):
        cm = cm.detach().cpu().numpy().astype(np.float64)
    else:
        cm = np.asarray(cm, dtype=np.float64)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    den_iou = tp + fp + fn
    iou = np.divide(tp, den_iou, out=np.full_like(tp, np.nan), where=den_iou > 0)

    den_dice = 2 * tp + fp + fn
    dice = np.divide(2 * tp, den_dice, out=np.full_like(tp, np.nan), where=den_dice > 0)

    accuracy = float(tp.sum() / max(cm.sum(), 1.0))

    out = {
        "mIoU": float(np.nanmean(iou)),
        "macro_dice": float(np.nanmean(dice)),
        "accuracy": accuracy,
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(CLASS_NAMES):
        out[f"{name}_IoU"] = float(iou[i])
        out[f"{name}_Dice"] = float(dice[i])

    return out

def compact_metrics(m):
    return {
        "mIoU": round(m["mIoU"], 4),
        "non_flood": round(m["non_flood_IoU"], 4),
        "permanent": round(m["permanent_water_IoU"], 4),
        "temporary": round(m["temporary_flood_IoU"], 4),
    }

def _tta_transform(tensor, mode):
    if mode == "identity":
        return tensor
    if mode == "hflip":
        return torch.flip(tensor, dims=[-1])
    if mode == "vflip":
        return torch.flip(tensor, dims=[-2])
    if mode == "rot180":
        return torch.rot90(tensor, k=2, dims=(-2, -1))
    raise ValueError(mode)

def _tta_inverse(tensor, mode):
    return _tta_transform(tensor, mode)

def calibrated_log_probs_from_aux(aux, type_bias):
    water_logit = aux["water_logit"].float()
    type_logit = aux["type_logit"].float() + float(type_bias)

    p_water = torch.sigmoid(water_logit)
    p_temp_given_water = torch.sigmoid(type_logit)

    p0 = 1.0 - p_water
    p1 = p_water * (1.0 - p_temp_given_water)
    p2 = p_water * p_temp_given_water

    probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
    return probs.log()

@torch.inference_mode()
def frozen_tta_log_probs(model_to_eval, sar, optical, reliability):
    total = None

    for mode in TTA_MODES:
        sar_t = _tta_transform(sar, mode)
        opt_t = _tta_transform(optical, mode)
        rel_t = _tta_transform(reliability, mode)

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            aux = model_to_eval(
                sar_t,
                opt_t,
                rel_t,
                return_aux=True,
            )

        log_probs = calibrated_log_probs_from_aux(aux, TYPE_BIAS)
        log_probs = _tta_inverse(log_probs, mode).float()

        total = log_probs if total is None else total + log_probs

    return total / len(TTA_MODES)

def probs_with_temperature(avg_log_probs, temperature):
    t = float(temperature)
    return torch.softmax(avg_log_probs.float() / t, dim=1)

print("P12-C08 COMPLETE ✅")


P12-C08 COMPLETE ✅


In [9]:
# ============================================================
# CELL: P12-C09 — CLEAN FULL-VALIDATION INTEGRITY PASS + CACHE
#
# One clean pass is cached so that temperature fitting and the clean
# robustness baseline do not require duplicate model inference.
# ============================================================
EXPECTED_FULL = {
    "mIoU": 0.796029,
    "non_flood_IoU": 0.976510,
    "permanent_water_IoU": 0.825058,
    "temporary_flood_IoU": 0.586519,
}

clean_cache = []
cm_full = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

start = time.time()
model.eval()

with torch.inference_mode():
    for bi, batch in enumerate(val_full_loader, start=1):
        sar = batch["sar"].to(DEVICE, non_blocking=True)
        opt = batch["optical"].to(DEVICE, non_blocking=True)
        rel = batch["reliability"].to(DEVICE, non_blocking=True)
        y = batch["target"].to(DEVICE, non_blocking=True)

        avg_log_probs = frozen_tta_log_probs(model, sar, opt, rel)
        pred = avg_log_probs.argmax(1)

        cm_full += confusion_from_predictions(pred, y).cpu()

        # Cache per tile on CPU. Float16 is sufficient for later scalar
        # temperature fitting / uncertainty statistics and keeps RAM modest.
        avg_lp_cpu = avg_log_probs.detach().cpu().half()
        y_cpu = y.detach().cpu().to(torch.int16)

        scene_ids = list(batch["scene_id"])
        tile_indices = batch["tile_index"]

        for i in range(avg_lp_cpu.shape[0]):
            clean_cache.append({
                "scene_id": str(scene_ids[i]),
                "tile_index": int(tile_indices[i]),
                "log_probs": avg_lp_cpu[i].clone(),
                "target": y_cpu[i].clone(),
            })

        if bi == 1 or bi % 30 == 0 or bi == len(val_full_loader):
            print(f"[{bi:03d}/{len(val_full_loader)}] clean integrity")

full_clean_metrics = metrics_from_confusion(cm_full)

print("=" * 112)
print("P12-C09 — FULL CLEAN INTEGRITY")
print("=" * 112)
print("Reproduced:", compact_metrics(full_clean_metrics))
print("Expected  :", {
    "mIoU": round(EXPECTED_FULL["mIoU"], 4),
    "non_flood": round(EXPECTED_FULL["non_flood_IoU"], 4),
    "permanent": round(EXPECTED_FULL["permanent_water_IoU"], 4),
    "temporary": round(EXPECTED_FULL["temporary_flood_IoU"], 4),
})
print("Cached tiles:", len(clean_cache))
print("Minutes     :", round((time.time() - start) / 60, 2))

for key, ref in EXPECTED_FULL.items():
    assert abs(full_clean_metrics[key] - ref) < 0.0015, (
        f"Frozen model reproduction failed for {key}: "
        f"{full_clean_metrics[key]} vs {ref}"
    )

assert len(clean_cache) == len(val_df) * TILES_PER_SCENE
print("P12-C09 COMPLETE ✅")


[001/136] clean integrity
[030/136] clean integrity
[060/136] clean integrity
[090/136] clean integrity
[120/136] clean integrity
[136/136] clean integrity
P12-C09 — FULL CLEAN INTEGRITY
Reproduced: {'mIoU': 0.796, 'non_flood': 0.9765, 'permanent': 0.8251, 'temporary': 0.5865}
Expected  : {'mIoU': 0.796, 'non_flood': 0.9765, 'permanent': 0.8251, 'temporary': 0.5865}
Cached tiles: 272
Minutes     : 0.69
P12-C09 COMPLETE ✅


In [10]:
# ============================================================
# CELL: P12-C10 — FIT ONE SCALAR TEMPERATURE ON CALIBRATION SCENES
#
# Temperature scaling changes confidence only; class argmax is invariant.
# ============================================================
cal_entries = [x for x in clean_cache if x["scene_id"] in calibration_ids]
robust_clean_entries = [x for x in clean_cache if x["scene_id"] in robustness_ids]

assert len(cal_entries) == 34 * TILES_PER_SCENE
assert len(robust_clean_entries) == 34 * TILES_PER_SCENE

# Build a deterministic natural-frequency pixel sample for scalar T fitting.
rng = np.random.default_rng(SEED + 1205)
SAMPLE_PER_TILE = 4096

sample_lp = []
sample_y = []

for entry in cal_entries:
    lp = entry["log_probs"].float().permute(1, 2, 0).reshape(-1, NUM_CLASSES)
    y = entry["target"].reshape(-1).long()

    valid_idx = torch.nonzero(y != IGNORE_INDEX, as_tuple=False).squeeze(1)
    if valid_idx.numel() == 0:
        continue

    take = min(SAMPLE_PER_TILE, valid_idx.numel())
    chosen_np = rng.choice(valid_idx.numel(), size=take, replace=False)
    chosen = valid_idx[torch.from_numpy(chosen_np).long()]

    sample_lp.append(lp[chosen])
    sample_y.append(y[chosen])

sample_lp = torch.cat(sample_lp, dim=0).to(DEVICE)
sample_y = torch.cat(sample_y, dim=0).to(DEVICE)

print("Temperature-fit sample pixels:", sample_y.numel())

log_T = nn.Parameter(torch.tensor(0.0, device=DEVICE))
optimizer_T = torch.optim.LBFGS(
    [log_T],
    lr=0.25,
    max_iter=60,
    tolerance_grad=1e-9,
    tolerance_change=1e-10,
    line_search_fn="strong_wolfe",
)

with torch.no_grad():
    nll_before = F.nll_loss(
        F.log_softmax(sample_lp / 1.0, dim=1),
        sample_y,
    ).item()

def closure():
    optimizer_T.zero_grad()
    T = torch.exp(log_T).clamp(0.05, 10.0)
    scaled_log_probs = F.log_softmax(sample_lp / T, dim=1)
    loss = F.nll_loss(scaled_log_probs, sample_y)
    loss.backward()
    return loss

optimizer_T.step(closure)

TEMPERATURE = float(torch.exp(log_T.detach()).clamp(0.05, 10.0).cpu())

with torch.no_grad():
    nll_after = F.nll_loss(
        F.log_softmax(sample_lp / TEMPERATURE, dim=1),
        sample_y,
    ).item()

print("=" * 112)
print("P12-C10 — TEMPERATURE SCALING")
print("=" * 112)
print("Temperature :", TEMPERATURE)
print("NLL before  :", nll_before)
print("NLL after   :", nll_after)
print("Improvement :", nll_before - nll_after)

assert np.isfinite(TEMPERATURE)
assert 0.05 <= TEMPERATURE <= 10.0
assert nll_after <= nll_before + 1e-7

del sample_lp, sample_y
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("P12-C10 COMPLETE ✅")


Temperature-fit sample pixels: 557056
P12-C10 — TEMPERATURE SCALING
Temperature : 0.989592432975769
NLL before  : 0.07632777094841003
NLL after   : 0.07631631195545197
Improvement : 1.1458992958068848e-05
P12-C10 COMPLETE ✅


In [11]:
# ============================================================
# CELL: P12-C11 — FREEZE CLEAN ENTROPY ABSTENTION THRESHOLDS
#
# Thresholds are derived ONLY from the calibration half.
# - REVIEW_10 threshold: top ~10% most uncertain pixels reviewed
# - REVIEW_20 threshold: top ~20% most uncertain pixels reviewed
# ============================================================
def normalized_entropy(probs):
    return -(probs * probs.clamp_min(1e-8).log()).sum(dim=1) / math.log(NUM_CLASSES)

entropy_chunks = []

for entry in cal_entries:
    lp = entry["log_probs"].float().unsqueeze(0)
    y = entry["target"].long().unsqueeze(0)

    probs = probs_with_temperature(lp, TEMPERATURE)
    ent = normalized_entropy(probs)[0]

    valid = y[0] != IGNORE_INDEX
    entropy_chunks.append(ent[valid].cpu().numpy().astype(np.float32))

cal_entropy = np.concatenate(entropy_chunks)

ENTROPY_THRESHOLD_REVIEW_10 = float(np.quantile(cal_entropy, 0.90))
ENTROPY_THRESHOLD_REVIEW_20 = float(np.quantile(cal_entropy, 0.80))

print("=" * 112)
print("P12-C11 — FROZEN ABSTENTION POLICY")
print("=" * 112)
print("Calibration valid pixels    :", len(cal_entropy))
print("10% review entropy threshold:", ENTROPY_THRESHOLD_REVIEW_10)
print("20% review entropy threshold:", ENTROPY_THRESHOLD_REVIEW_20)

assert 0.0 <= ENTROPY_THRESHOLD_REVIEW_20 <= 1.0
assert 0.0 <= ENTROPY_THRESHOLD_REVIEW_10 <= 1.0
assert ENTROPY_THRESHOLD_REVIEW_20 <= ENTROPY_THRESHOLD_REVIEW_10

print("P12-C11 COMPLETE ✅")


P12-C11 — FROZEN ABSTENTION POLICY
Calibration valid pixels    : 7700779
10% review entropy threshold: 0.1218614846467972
20% review entropy threshold: 0.04920305460691452
P12-C11 COMPLETE ✅


In [12]:
# ============================================================
# CELL: P12-C12 — UNCERTAINTY METRICS / STREAMING ACCUMULATOR
# ============================================================
ECE_BINS = 15

class UncertaintyAccumulator:
    def __init__(self):
        self.cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
        self.cm_review10 = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
        self.cm_review20 = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

        self.n = 0
        self.sum_nll = 0.0
        self.sum_brier = 0.0
        self.sum_conf = 0.0
        self.sum_entropy = 0.0

        self.correct_entropy_sum = 0.0
        self.correct_n = 0
        self.error_entropy_sum = 0.0
        self.error_n = 0

        self.review10_n = 0
        self.review20_n = 0

        self.ece_count = np.zeros(ECE_BINS, dtype=np.int64)
        self.ece_conf_sum = np.zeros(ECE_BINS, dtype=np.float64)
        self.ece_acc_sum = np.zeros(ECE_BINS, dtype=np.float64)

        self.error_scores = []
        self.error_labels = []

    def update(self, probs, target):
        valid = target != IGNORE_INDEX
        if not valid.any():
            return

        probs_v = probs.permute(0, 2, 3, 1)[valid]
        target_v = target[valid].long()

        conf, pred = probs_v.max(dim=1)
        correct = pred.eq(target_v)

        ent = -(probs_v * probs_v.clamp_min(1e-8).log()).sum(dim=1) / math.log(NUM_CLASSES)

        true_p = probs_v.gather(1, target_v[:, None]).squeeze(1).clamp_min(1e-8)
        nll = -true_p.log()

        onehot = F.one_hot(target_v, num_classes=NUM_CLASSES).float()
        brier = ((probs_v - onehot) ** 2).sum(dim=1)

        self.cm += confusion_from_predictions(
            pred.reshape(-1),
            target_v.reshape(-1),
        ).cpu()

        n = int(target_v.numel())
        self.n += n
        self.sum_nll += float(nll.sum().item())
        self.sum_brier += float(brier.sum().item())
        self.sum_conf += float(conf.sum().item())
        self.sum_entropy += float(ent.sum().item())

        if correct.any():
            self.correct_entropy_sum += float(ent[correct].sum().item())
            self.correct_n += int(correct.sum().item())

        err = ~correct
        if err.any():
            self.error_entropy_sum += float(ent[err].sum().item())
            self.error_n += int(err.sum().item())

        # ECE bins.
        bin_idx = torch.clamp((conf * ECE_BINS).long(), max=ECE_BINS - 1)
        for b in range(ECE_BINS):
            m = bin_idx == b
            if m.any():
                self.ece_count[b] += int(m.sum().item())
                self.ece_conf_sum[b] += float(conf[m].sum().item())
                self.ece_acc_sum[b] += float(correct[m].float().sum().item())

        # Fixed abstention / review policies.
        keep10 = ent <= ENTROPY_THRESHOLD_REVIEW_10
        keep20 = ent <= ENTROPY_THRESHOLD_REVIEW_20

        self.review10_n += int((~keep10).sum().item())
        self.review20_n += int((~keep20).sum().item())

        if keep10.any():
            self.cm_review10 += confusion_from_predictions(
                pred[keep10],
                target_v[keep10],
            ).cpu()

        if keep20.any():
            self.cm_review20 += confusion_from_predictions(
                pred[keep20],
                target_v[keep20],
            ).cpu()

        # Keep full robustness-half error/uncertainty scores for AUROC/AUPRC.
        self.error_scores.append(ent.detach().cpu().numpy().astype(np.float32))
        self.error_labels.append(err.detach().cpu().numpy().astype(np.uint8))

    def finalize(self):
        base = metrics_from_confusion(self.cm)

        ece = 0.0
        for b in range(ECE_BINS):
            if self.ece_count[b] == 0:
                continue
            avg_conf = self.ece_conf_sum[b] / self.ece_count[b]
            avg_acc = self.ece_acc_sum[b] / self.ece_count[b]
            ece += (self.ece_count[b] / max(self.n, 1)) * abs(avg_conf - avg_acc)

        scores = np.concatenate(self.error_scores) if self.error_scores else np.array([])
        labels = np.concatenate(self.error_labels) if self.error_labels else np.array([])

        if labels.size and np.unique(labels).size == 2:
            error_auroc = float(roc_auc_score(labels, scores))
            error_auprc = float(average_precision_score(labels, scores))
        else:
            error_auroc = float("nan")
            error_auprc = float("nan")

        review10_metrics = metrics_from_confusion(self.cm_review10)
        review20_metrics = metrics_from_confusion(self.cm_review20)

        out = dict(base)
        out.update({
            "n_valid": int(self.n),
            "nll": self.sum_nll / max(self.n, 1),
            "brier": self.sum_brier / max(self.n, 1),
            "ece_15": float(ece),
            "mean_confidence": self.sum_conf / max(self.n, 1),
            "mean_entropy": self.sum_entropy / max(self.n, 1),
            "mean_entropy_correct": self.correct_entropy_sum / max(self.correct_n, 1),
            "mean_entropy_error": self.error_entropy_sum / max(self.error_n, 1),
            "entropy_error_auroc": error_auroc,
            "entropy_error_auprc": error_auprc,
            "review10_rate": self.review10_n / max(self.n, 1),
            "review20_rate": self.review20_n / max(self.n, 1),
            "retained10_mIoU": review10_metrics["mIoU"],
            "retained20_mIoU": review20_metrics["mIoU"],
            "retained10_accuracy": review10_metrics["accuracy"],
            "retained20_accuracy": review20_metrics["accuracy"],
        })
        return out

print("P12-C12 COMPLETE ✅")


P12-C12 COMPLETE ✅


In [13]:
# ============================================================
# CELL: P12-C13 — CLEAN ROBUSTNESS-HALF CALIBRATION CHECK
#
# Uses cached clean logits; no model inference required.
# Reports confidence calibration BEFORE and AFTER scalar temperature scaling.
# ============================================================
def evaluate_cached_entries(entries, temperature):
    acc = UncertaintyAccumulator()

    for entry in entries:
        lp = entry["log_probs"].float().unsqueeze(0)
        y = entry["target"].long().unsqueeze(0)
        probs = probs_with_temperature(lp, temperature)
        acc.update(probs, y)

    return acc.finalize()

clean_uncal = evaluate_cached_entries(robust_clean_entries, temperature=1.0)
clean_cal = evaluate_cached_entries(robust_clean_entries, temperature=TEMPERATURE)

print("=" * 112)
print("P12-C13 — CLEAN ROBUSTNESS-HALF CONFIDENCE")
print("=" * 112)
print("Segmentation:", compact_metrics(clean_cal))
print()
print("Before temperature:")
print("  NLL   :", clean_uncal["nll"])
print("  Brier :", clean_uncal["brier"])
print("  ECE15 :", clean_uncal["ece_15"])
print()
print("After temperature:")
print("  T     :", TEMPERATURE)
print("  NLL   :", clean_cal["nll"])
print("  Brier :", clean_cal["brier"])
print("  ECE15 :", clean_cal["ece_15"])
print("  Err AUROC:", clean_cal["entropy_error_auroc"])
print("  Err AUPRC:", clean_cal["entropy_error_auprc"])
print("  10% review actual rate:", clean_cal["review10_rate"])
print("  20% review actual rate:", clean_cal["review20_rate"])
print("  retained mIoU @10% policy:", clean_cal["retained10_mIoU"])
print("  retained mIoU @20% policy:", clean_cal["retained20_mIoU"])

# Temperature scaling should not alter segmentation predictions.
assert abs(clean_uncal["mIoU"] - clean_cal["mIoU"]) < 1e-12

print("P12-C13 COMPLETE ✅")


P12-C13 — CLEAN ROBUSTNESS-HALF CONFIDENCE
Segmentation: {'mIoU': 0.8265, 'non_flood': 0.9706, 'permanent': 0.8885, 'temporary': 0.6204}

Before temperature:
  NLL   : 0.09802478051609359
  Brier : 0.05000974376045874
  ECE15 : 0.002340094834978993

After temperature:
  T     : 0.989592432975769
  NLL   : 0.09798401492272979
  Brier : 0.050020136112905476
  ECE15 : 0.002112175587095098
  Err AUROC: 0.9386093497845992
  Err AUPRC: 0.3439611462456254
  10% review actual rate: 0.15641681246408526
  20% review actual rate: 0.2771040816163541
  retained mIoU @10% policy: 0.9437445816720276
  retained mIoU @20% policy: 0.9610201658069958
P12-C13 COMPLETE ✅


In [14]:
# ============================================================
# CELL: P12-C14 — PREDECLARED ROBUSTNESS SCENARIOS
# ============================================================
SCENARIOS = [
    {
        "name": "optical_haze_proxy",
        "description": "7x7 low-pass optical blur; contrast attenuated to 0.60; reliability x0.25",
    },
    {
        "name": "optical_missing",
        "description": "optical standardized tensor set to 0; reliability set to 0",
    },
    {
        "name": "sar_noise_0p35",
        "description": "Gaussian standardized-domain SAR stress sigma=0.35 on nonzero SAR pixels",
    },
    {
        "name": "sar_missing",
        "description": "SAR standardized tensor set to 0",
    },
    {
        "name": "combined_mild",
        "description": "optical haze proxy + Gaussian SAR stress sigma=0.25",
    },
    {
        "name": "reliability_neutral",
        "description": "P(clear) forced to 0.5 everywhere",
    },
    {
        "name": "reliability_inverted",
        "description": "P(clear) replaced by 1-P(clear); adversarial reliability stress",
    },
]

SCENARIO_SEEDS = {
    name: 51000 + i * 1000
    for i, name in enumerate([s["name"] for s in SCENARIOS], start=1)
}

def _deterministic_noise_like(tensor, seed):
    g = torch.Generator(device=tensor.device)
    g.manual_seed(int(seed))
    return torch.randn(
        tensor.shape,
        dtype=tensor.dtype,
        device=tensor.device,
        generator=g,
    )

def apply_robustness_scenario(sar, optical, reliability, scenario_name, batch_index):
    sar_c = sar.clone()
    opt_c = optical.clone()
    rel_c = reliability.clone()

    if scenario_name == "clean":
        return sar_c, opt_c, rel_c

    if scenario_name == "optical_haze_proxy":
        blur = F.avg_pool2d(opt_c, kernel_size=7, stride=1, padding=3)
        opt_c = 0.60 * blur
        rel_c = 0.25 * rel_c
        return sar_c, opt_c, rel_c

    if scenario_name == "optical_missing":
        opt_c.zero_()
        rel_c.zero_()
        return sar_c, opt_c, rel_c

    if scenario_name == "sar_noise_0p35":
        seed = SCENARIO_SEEDS[scenario_name] + int(batch_index)
        noise = _deterministic_noise_like(sar_c, seed)
        sar_valid = (sar_c.abs().sum(dim=1, keepdim=True) > 0).to(sar_c.dtype)
        sar_c = sar_c + 0.35 * noise * sar_valid
        return sar_c, opt_c, rel_c

    if scenario_name == "sar_missing":
        sar_c.zero_()
        return sar_c, opt_c, rel_c

    if scenario_name == "combined_mild":
        blur = F.avg_pool2d(opt_c, kernel_size=7, stride=1, padding=3)
        opt_c = 0.60 * blur
        rel_c = 0.25 * rel_c

        seed = SCENARIO_SEEDS[scenario_name] + int(batch_index)
        noise = _deterministic_noise_like(sar_c, seed)
        sar_valid = (sar_c.abs().sum(dim=1, keepdim=True) > 0).to(sar_c.dtype)
        sar_c = sar_c + 0.25 * noise * sar_valid
        return sar_c, opt_c, rel_c

    if scenario_name == "reliability_neutral":
        rel_c.fill_(0.5)
        return sar_c, opt_c, rel_c

    if scenario_name == "reliability_inverted":
        rel_c = 1.0 - rel_c
        return sar_c, opt_c, rel_c

    raise ValueError(f"Unknown robustness scenario: {scenario_name}")

print("=" * 112)
print("P12-C14 — FROZEN ROBUSTNESS PROTOCOL")
print("=" * 112)
for s in SCENARIOS:
    print(f"{s['name']:<24} : {s['description']}")
print("P12-C14 COMPLETE ✅")


P12-C14 — FROZEN ROBUSTNESS PROTOCOL
optical_haze_proxy       : 7x7 low-pass optical blur; contrast attenuated to 0.60; reliability x0.25
optical_missing          : optical standardized tensor set to 0; reliability set to 0
sar_noise_0p35           : Gaussian standardized-domain SAR stress sigma=0.35 on nonzero SAR pixels
sar_missing              : SAR standardized tensor set to 0
combined_mild            : optical haze proxy + Gaussian SAR stress sigma=0.25
reliability_neutral      : P(clear) forced to 0.5 everywhere
reliability_inverted     : P(clear) replaced by 1-P(clear); adversarial reliability stress
P12-C14 COMPLETE ✅


In [15]:
# ============================================================
# CELL: P12-C15 — ROBUSTNESS + UNCERTAINTY EVALUATION
# ============================================================
@torch.inference_mode()
def evaluate_scenario(scenario_name):
    acc = UncertaintyAccumulator()
    model.eval()

    for bi, batch in enumerate(val_robust_loader, start=1):
        sar = batch["sar"].to(DEVICE, non_blocking=True)
        opt = batch["optical"].to(DEVICE, non_blocking=True)
        rel = batch["reliability"].to(DEVICE, non_blocking=True)
        y = batch["target"].to(DEVICE, non_blocking=True)

        sar_c, opt_c, rel_c = apply_robustness_scenario(
            sar,
            opt,
            rel,
            scenario_name,
            batch_index=bi,
        )

        avg_log_probs = frozen_tta_log_probs(
            model,
            sar_c,
            opt_c,
            rel_c,
        )

        probs = probs_with_temperature(avg_log_probs, TEMPERATURE)
        acc.update(probs, y)

        if bi == 1 or bi % 20 == 0 or bi == len(val_robust_loader):
            print(f"[{bi:03d}/{len(val_robust_loader)}] {scenario_name}")

    return acc.finalize()

results = []

# Clean scenario comes from the already-cached exact clean inference.
clean_row = dict(clean_cal)
clean_row["scenario"] = "clean"
clean_row["description"] = "unchanged frozen input"
results.append(clean_row)

start_all = time.time()

for s in SCENARIOS:
    print("\n" + "=" * 112)
    print("SCENARIO:", s["name"])
    print("=" * 112)

    t0 = time.time()
    m = evaluate_scenario(s["name"])
    m["scenario"] = s["name"]
    m["description"] = s["description"]
    m["minutes"] = (time.time() - t0) / 60.0
    results.append(m)

    print("Segmentation :", compact_metrics(m))
    print("ECE15        :", round(m["ece_15"], 6))
    print("Brier        :", round(m["brier"], 6))
    print("Entropy      :", round(m["mean_entropy"], 6))
    print("Error AUROC  :", round(m["entropy_error_auroc"], 6))
    print("Review10 rate:", round(m["review10_rate"], 4))
    print("Review20 rate:", round(m["review20_rate"], 4))
    print("Minutes      :", round(m["minutes"], 2))

print("\nTotal robustness minutes:", round((time.time() - start_all) / 60, 2))

results_df = pd.DataFrame(results)

clean_miou = float(results_df.loc[results_df["scenario"] == "clean", "mIoU"].iloc[0])
clean_temp = float(results_df.loc[results_df["scenario"] == "clean", "temporary_flood_IoU"].iloc[0])

results_df["delta_mIoU_vs_clean"] = results_df["mIoU"] - clean_miou
results_df["delta_temp_IoU_vs_clean"] = results_df["temporary_flood_IoU"] - clean_temp
results_df["relative_mIoU_retention"] = results_df["mIoU"] / max(clean_miou, 1e-12)

display_cols = [
    "scenario",
    "mIoU",
    "non_flood_IoU",
    "permanent_water_IoU",
    "temporary_flood_IoU",
    "delta_mIoU_vs_clean",
    "relative_mIoU_retention",
    "ece_15",
    "brier",
    "mean_entropy",
    "entropy_error_auroc",
    "review10_rate",
    "review20_rate",
    "retained10_mIoU",
    "retained20_mIoU",
]

print("\nROBUSTNESS + UNCERTAINTY SCORECARD")
display(
    results_df[display_cols].style.format({
        c: "{:.6f}"
        for c in display_cols
        if c != "scenario"
    })
)

results_csv = METRIC_ROOT / "FloodShift_05_robustness_uncertainty.csv"
results_df.to_csv(results_csv, index=False)

print("Saved:", results_csv)
print("P12-C15 COMPLETE ✅")



SCENARIO: optical_haze_proxy
[001/68] optical_haze_proxy
[020/68] optical_haze_proxy
[040/68] optical_haze_proxy
[060/68] optical_haze_proxy
[068/68] optical_haze_proxy
Segmentation : {'mIoU': 0.5003, 'non_flood': 0.9062, 'permanent': 0.3847, 'temporary': 0.2099}
ECE15        : 0.067118
Brier        : 0.158401
Entropy      : 0.084049
Error AUROC  : 0.943427
Review10 rate: 0.1234
Review20 rate: 0.1904
Minutes      : 0.42

SCENARIO: optical_missing
[001/68] optical_missing
[020/68] optical_missing
[040/68] optical_missing
[060/68] optical_missing
[068/68] optical_missing
Segmentation : {'mIoU': 0.2917, 'non_flood': 0.8602, 'permanent': 0.0148, 'temporary': 0.0}
ECE15        : 0.132226
Brier        : 0.271098
Entropy      : 0.035081
Error AUROC  : 0.746655
Review10 rate: 0.02
Review20 rate: 0.0617
Minutes      : 0.39

SCENARIO: sar_noise_0p35
[001/68] sar_noise_0p35
[020/68] sar_noise_0p35
[040/68] sar_noise_0p35
[060/68] sar_noise_0p35
[068/68] sar_noise_0p35
Segmentation : {'mIoU': 0.8

,scenario,mIoU,non_flood_IoU,permanent_water_IoU,temporary_flood_IoU,delta_mIoU_vs_clean,relative_mIoU_retention,ece_15,brier,mean_entropy,entropy_error_auroc,review10_rate,review20_rate,retained10_mIoU,retained20_mIoU
0,clean,0.826512,0.970587,0.888512,0.620438,0.000000,1.000000,0.002112,0.050020,0.091689,0.938609,0.156417,0.277104,0.943745,0.961020
1,optical_haze_proxy,0.500262,0.906187,0.384713,0.209887,-0.326250,0.605269,0.067118,0.158401,0.084049,0.943427,0.123378,0.190450,0.440030,0.482680
2,optical_missing,0.291677,0.860238,0.014795,0.000000,-0.534835,0.352901,0.132226,0.271098,0.035081,0.746655,0.020029,0.061739,0.291433,0.298249
3,sar_noise_0p35,0.854220,0.972864,0.928945,0.660852,0.027708,1.033524,0.003678,0.042232,0.088369,0.945083,0.148357,0.262495,0.960497,0.966090
4,sar_missing,0.560866,0.951219,0.302619,0.428760,-0.265647,0.678593,0.038229,0.118513,0.107470,0.946056,0.170711,0.253662,0.648065,0.722508
5,combined_mild,0.458744,0.893417,0.297177,0.185638,-0.367769,0.555036,0.081022,0.180772,0.074383,0.947905,0.106871,0.178179,0.383017,0.421496
6,reliability_neutral,0.819581,0.967433,0.892244,0.599066,-0.006932,0.991614,0.002736,0.054226,0.093386,0.933988,0.161696,0.283430,0.935703,0.948148
7,reliability_inverted,0.797014,0.965029,0.857673,0.568341,-0.029498,0.964310,0.003838,0.060822,0.100057,0.926934,0.170297,0.292846,0.904154,0.913233


Saved: /kaggle/working/FloodShift_05_Robustness_Uncertainty/metrics/FloodShift_05_robustness_uncertainty.csv
P12-C15 COMPLETE ✅


In [17]:
# ============================================================
# CELL: P12-C16 — SUMMARY DIAGNOSTICS + FREEZE UNCERTAINTY POLICY
# ============================================================
clean = results_df[results_df["scenario"] == "clean"].iloc[0]

summary_rows = []
for _, row in results_df.iterrows():
    summary_rows.append({
        "scenario": row["scenario"],
        "mIoU": float(row["mIoU"]),
        "mIoU_drop": float(clean["mIoU"] - row["mIoU"]),
        "temporary_IoU": float(row["temporary_flood_IoU"]),
        "mean_entropy": float(row["mean_entropy"]),
        "error_AUROC": float(row["entropy_error_auroc"]),
        "review10_rate": float(row["review10_rate"]),
        "retained10_mIoU": float(row["retained10_mIoU"]),
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 112)
print("P12-C16 — ROBUSTNESS INTERPRETATION TABLE")
print("=" * 112)
display(summary_df.style.format({
    "mIoU": "{:.4f}",
    "mIoU_drop": "{:.4f}",
    "temporary_IoU": "{:.4f}",
    "mean_entropy": "{:.4f}",
    "error_AUROC": "{:.4f}",
    "review10_rate": "{:.4f}",
    "retained10_mIoU": "{:.4f}",
}))

# Diagnostic flags: these characterize the system; they do NOT reopen segmentation tuning.
diagnostic = {
    "temperature_improved_clean_robust_half_nll":
        bool(clean_cal["nll"] <= clean_uncal["nll"] + 1e-7),
    "temperature_improved_clean_robust_half_ece":
        bool(clean_cal["ece_15"] <= clean_uncal["ece_15"] + 1e-4),
    "clean_entropy_error_auroc_ge_0p70":
        bool(clean_cal["entropy_error_auroc"] >= 0.70),
    "clean_review10_improves_mIoU":
        bool(clean_cal["retained10_mIoU"] >= clean_cal["mIoU"]),
    "clean_review20_improves_mIoU":
        bool(clean_cal["retained20_mIoU"] >= clean_cal["mIoU"]),
}

print("\nDIAGNOSTIC FLAGS")
for k, v in diagnostic.items():
    print(f"{k:<52}: {'✅' if v else '⚠️'}")

print("\nSegmentation weights/type bias remain FROZEN regardless of these diagnostics.")
print("P12-C16 COMPLETE ✅")


P12-C16 — ROBUSTNESS INTERPRETATION TABLE


,scenario,mIoU,mIoU_drop,temporary_IoU,mean_entropy,error_AUROC,review10_rate,retained10_mIoU
0,clean,0.8265,0.0000,0.6204,0.0917,0.9386,0.1564,0.9437
1,optical_haze_proxy,0.5003,0.3263,0.2099,0.0840,0.9434,0.1234,0.4400
2,optical_missing,0.2917,0.5348,0.0000,0.0351,0.7467,0.0200,0.2914
3,sar_noise_0p35,0.8542,-0.0277,0.6609,0.0884,0.9451,0.1484,0.9605
4,sar_missing,0.5609,0.2656,0.4288,0.1075,0.9461,0.1707,0.6481
5,combined_mild,0.4587,0.3678,0.1856,0.0744,0.9479,0.1069,0.3830
6,reliability_neutral,0.8196,0.0069,0.5991,0.0934,0.9340,0.1617,0.9357
7,reliability_inverted,0.7970,0.0295,0.5683,0.1001,0.9269,0.1703,0.9042



DIAGNOSTIC FLAGS
temperature_improved_clean_robust_half_nll          : ✅
temperature_improved_clean_robust_half_ece          : ✅
clean_entropy_error_auroc_ge_0p70                   : ✅
clean_review10_improves_mIoU                        : ✅
clean_review20_improves_mIoU                        : ✅

Segmentation weights/type bias remain FROZEN regardless of these diagnostics.
P12-C16 COMPLETE ✅


In [18]:
# ============================================================
# CELL: P12-C17 — EXPORT FROZEN ROBUSTNESS / UNCERTAINTY PROTOCOL
# ============================================================
uncertainty_config = {
    "experiment": "FloodShift-05-Robustness-Uncertainty",
    "segmenter": "FloodShift-04H-CAL",
    "source_04h_sha256": SHA_04H,
    "type_bias": TYPE_BIAS,
    "temperature": TEMPERATURE,
    "entropy_threshold_review_10": ENTROPY_THRESHOLD_REVIEW_10,
    "entropy_threshold_review_20": ENTROPY_THRESHOLD_REVIEW_20,
    "uncertainty_score": "normalized_predictive_entropy",
    "temperature_fit_partition": "deterministic 34-scene validation calibration half",
    "robustness_partition": "disjoint deterministic 34-scene validation robustness half",
    "tta_modes": TTA_MODES,
    "ece_bins": ECE_BINS,
    "test_status": "LOCKED_NOT_EVALUATED",
}

protocol = {
    "experiment": "FloodShift-05-Robustness-Uncertainty",
    "scientific_role": (
        "post-segmentation robustness characterization, scalar confidence calibration, "
        "and fixed entropy-based abstention policy"
    ),
    "network_training_performed": False,
    "segmentation_model_changed": False,
    "full_clean_validation_integrity": full_clean_metrics,
    "clean_robustness_half_uncalibrated": clean_uncal,
    "clean_robustness_half_temperature_calibrated": clean_cal,
    "uncertainty_config": uncertainty_config,
    "scenario_definitions": SCENARIOS,
    "diagnostic_flags": diagnostic,
    "test_status": "LOCKED_NOT_EVALUATED",
}

config_path = OUTPUT_ROOT / "FloodShift_05_uncertainty_config.pt"
torch.save(uncertainty_config, config_path)

protocol_path = OUTPUT_ROOT / "FloodShift_05_protocol_record.json"
with open(protocol_path, "w") as f:
    json.dump(protocol, f, indent=2)

summary_path = METRIC_ROOT / "FloodShift_05_summary.csv"
summary_df.to_csv(summary_path, index=False)

archive_path = OUTPUT_ROOT / "FloodShift_05_ARCHIVE.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [
        config_path,
        protocol_path,
        METRIC_ROOT / "FloodShift_05_robustness_uncertainty.csv",
        summary_path,
    ]:
        if Path(p).exists():
            zf.write(p, arcname=Path(p).name)

print("=" * 112)
print("P12-C17 — EXPORT")
print("=" * 112)
print("Uncertainty config :", config_path)
print("Protocol record    :", protocol_path)
print("Full scorecard     :", METRIC_ROOT / "FloodShift_05_robustness_uncertainty.csv")
print("Summary CSV        :", summary_path)
print("Compact archive    :", archive_path)
print()
print("04H-CAL weights    : FROZEN")
print("Type bias          :", TYPE_BIAS, "(FROZEN)")
print("Temperature        :", TEMPERATURE, "(NOW FROZEN)")
print("Review-10 entropy  :", ENTROPY_THRESHOLD_REVIEW_10, "(NOW FROZEN)")
print("Review-20 entropy  :", ENTROPY_THRESHOLD_REVIEW_20, "(NOW FROZEN)")
print("Frozen geo test    : LOCKED_NOT_EVALUATED")
print()
print("NEXT STAGE: counterfactual SAR-vs-optical attribution.")
print("P12-C17 COMPLETE ✅")


P12-C17 — EXPORT
Uncertainty config : /kaggle/working/FloodShift_05_Robustness_Uncertainty/FloodShift_05_uncertainty_config.pt
Protocol record    : /kaggle/working/FloodShift_05_Robustness_Uncertainty/FloodShift_05_protocol_record.json
Full scorecard     : /kaggle/working/FloodShift_05_Robustness_Uncertainty/metrics/FloodShift_05_robustness_uncertainty.csv
Summary CSV        : /kaggle/working/FloodShift_05_Robustness_Uncertainty/metrics/FloodShift_05_summary.csv
Compact archive    : /kaggle/working/FloodShift_05_Robustness_Uncertainty/FloodShift_05_ARCHIVE.zip

04H-CAL weights    : FROZEN
Type bias          : -0.15 (FROZEN)
Temperature        : 0.989592432975769 (NOW FROZEN)
Review-10 entropy  : 0.1218614846467972 (NOW FROZEN)
Review-20 entropy  : 0.04920305460691452 (NOW FROZEN)
Frozen geo test    : LOCKED_NOT_EVALUATED

NEXT STAGE: counterfactual SAR-vs-optical attribution.
P12-C17 COMPLETE ✅


## After this notebook

Send the complete outputs from:

- **P12-C09** — clean 04H-CAL integrity reproduction
- **P12-C10** — fitted temperature and NLL change
- **P12-C11** — frozen entropy review thresholds
- **P12-C13** — clean calibration / error-detection metrics
- **P12-C15** — full robustness scorecard
- **P12-C16** — diagnostic flags
- **P12-C17** — exported artifacts

Do **not** reopen segmentation training based on these stress-test results.

Notebook 05 is designed to characterize failure modes and create a calibrated **review-required** mechanism. The next notebook uses the frozen model + uncertainty configuration to measure **counterfactual sensor contribution**:

\[
\Delta_{\mathrm{SAR}}
=
D(P_{\mathrm{full}}, P_{\mathrm{without\ SAR}})
\]

\[
\Delta_{\mathrm{OPT}}
=
D(P_{\mathrm{full}}, P_{\mathrm{without\ optical}})
\]

That attribution evidence will later become part of the grounded evidence packet consumed by the VLM/LLM.
